In [29]:
import json
import os
import re 
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback
import evaluate
from seqeval.metrics import classification_report
import seaborn as sns



INPUT_FILE_PATH = "./NER_dataset_multi.json" 
OUTPUT_FILE_PATH = "./training_data_bio_multi.json" 



In [30]:
def convert_to_bio(task_data: Dict) -> List[Tuple[List[str], List[str]]]:
    """
    Converte una singola task di Label Studio nel formato token/tag BIO.
    
    task_data: Un dizionario che rappresenta una singola riga dal JSON esportato.
    Ritorna: Una lista di coppie (tokens, tags) pronte per l'addestramento.
    """
    all_tasks_bio = []
    
    text = task_data['data']['text']
    annotations = task_data.get('annotations', [{}])[0].get('result', [])
    
    tokens: List[str] = text.split()
    token_tags: List[str] = ['O'] * len(tokens)
    
    current_char_pos = 0
    token_spans = [] 
    for token in tokens:
        match = re.search(re.escape(token), text[current_char_pos:])
        if match:
            start = current_char_pos + match.start()
            end = start + len(token)
            token_spans.append((start, end))
            current_char_pos = end
        else:
            token_spans.append((-1, -1))

    for region in annotations:
        if region['type'] == 'labels':
            start_char = region['value']['start']
            end_char = region['value']['end']
            label = region['value']['labels'][0]
            
            for i, (token_start, token_end) in enumerate(token_spans):
                
                if token_end > start_char and token_start < end_char:
                    
                    # (B)?
                    if token_start >= start_char:
                        if i == 0 or token_tags[i-1] == 'O':
                             token_tags[i] = f'B-{label}'
                        else: # (I)
                             token_tags[i] = f'I-{label}'
                             
                    elif token_tags[i] == 'O':
                        token_tags[i] = f'I-{label}'

    if tokens:
        all_tasks_bio.append((tokens, token_tags))
        
    return all_tasks_bio

In [31]:
bio_data: List[Tuple[List[str], List[str]]] = []

try:
    with open(INPUT_FILE_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    print(f"ERRORE: File non trovato al percorso: {INPUT_FILE_PATH}")
    data = []

for task in data:
    results = convert_to_bio(task)
    bio_data.extend(results)

if bio_data:
    print(f"Conversione completata. Numero totale di frasi processate: {len(bio_data)}")
    print("\nEsempio di output BIO (Token | Tag):")
    for token, tag in zip(bio_data[0][0], bio_data[0][1]):
        print(f"{token.ljust(15)} | {tag}")
else:
    print("path error")

with open(OUTPUT_FILE_PATH, 'w', encoding='utf-8') as f:
    json.dump(bio_data, f, indent=2)
    
print(f"\nSaved in: {OUTPUT_FILE_PATH}")

Conversione completata. Numero totale di frasi processate: 556

Esempio di output BIO (Token | Tag):
I’m             | O
thinking        | O
about           | O
diving          | O
deeper          | O
into            | O
Kotlin,         | B-INCLUDE_SKILL
but             | O
honestly        | O
I’d             | O
like            | O
to              | O
avoid           | O
C++             | B-AVOID_SKILL
for             | O
now             | O
because         | O
it              | O
stresses        | O
me              | O
out.            | O

Saved in: ./training_data_bio_multi.json


In [32]:
# --- Configuration ---
MODEL_CHECKPOINT = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# The path where you saved your BIO data from Cell 3
INPUT_FILE_PATH = "training_data_bio_multi.json" 
TEST_SET_SIZE = 0.1 # 10% of data for validation/test set

# --- Load and Prepare Data ---
with open(INPUT_FILE_PATH, 'r', encoding='utf-8') as f:
    # The JSON structure is [ [tokens], [tags] ]
    raw_data = json.load(f)

# Convert the list of (tokens, tags) tuples into a Hugging Face Dataset format
# We assume the conversion script outputted a list of lists: [[tokens_1, tags_1], [tokens_2, tags_2], ...]
data_dict = {
    'tokens': [item[0] for item in raw_data],
    'ner_tags_str': [item[1] for item in raw_data] # Temporarily store tags as strings
}

# Create a Dataset object
raw_dataset = Dataset.from_dict(data_dict)

# Split data into training and validation sets
train_test_split = raw_dataset.train_test_split(test_size=TEST_SET_SIZE)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

train_val_split = train_dataset.train_test_split(test_size=0.1)
val_dataset = train_val_split['test']
train_dataset = train_val_split['train']

print(f"Total samples loaded: {len(raw_dataset)}")
print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}, test samples: {len(test_dataset)}")

Total samples loaded: 556
Training samples: 450, Validation samples: 50, test samples: 56


In [33]:
# Cell 5: Tag Mapping and Tokenization

# 1. Define the actual labels used in your BIO format
# O (Outside), B-SKILL (Begin-SKILL), I-SKILL (Inside-SKILL)
# Get all unique tags from your loaded data
all_tags = set(tag for sample in raw_dataset['ner_tags_str'] for tag in sample)
label_list = sorted(list(all_tags))
label_map = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label_map.items()} # For output mapping

print("Label Mapping:")
print(label_map)

# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# 3. Define Alignment Function
def tokenize_and_align_labels(examples):
    # Tokenize the words (not the raw text, but the already split words)
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label_str_list in enumerate(examples["ner_tags_str"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens (like [CLS], [SEP]) get -100 tag
            if word_idx is None:
                label_ids.append(-100) 
            # Sub-word tokens (part of a word) should keep the tag of the main word
            elif word_idx != previous_word_idx:
                # Assign the original tag's ID (converted from string to int)
                label_ids.append(label_map[label_str_list[word_idx]])
            else:
                # If it's the second or subsequent sub-word piece of the same word, 
                # we keep the previous tag, but change 'B-' to 'I-' if applicable.
                # A simpler approach is to set sub-word tags to -100 to ignore them in loss calculation.
                # Here, we keep the tag ID for demonstration, but if it's the continuation of a word, 
                # it's usually set to -100 or the same tag as the first piece.
                label_ids.append(-100) # Setting to -100 is standard practice for sub-word pieces after the first.
                
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 4. Apply Tokenization and Alignment to the entire dataset
tokenized_train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_and_align_labels, batched=True)

print("Tokenization and alignment complete.")

Label Mapping:
{'B-ACQUIRED_SKILL': 0, 'B-AVOID_SKILL': 1, 'B-INCLUDE_SKILL': 2, 'B-NEUTRAL_SKILL': 3, 'I-ACQUIRED_SKILL': 4, 'I-AVOID_SKILL': 5, 'I-INCLUDE_SKILL': 6, 'I-NEUTRAL_SKILL': 7, 'O': 8}


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Tokenization and alignment complete.


In [34]:
# Cell 6: Model and Metrics Setup

# 1. Load Model
# AutoModelForTokenClassification adds a classification head on top of MiniLM
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, 
    num_labels=len(label_list), # Set the output size to the number of unique tags
    id2label=id2label,
    label2id=label_map
)

# 2. Define Evaluation Metrics
# We use the seqeval metric standard for NER tasks
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) and convert IDs back to string labels
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Calculate metrics
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Some weights of BertForTokenClassification were not initialized from the model checkpoint at sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [35]:
# Cell 7: Training Configuration and Execution

# 1. Define Training Arguments
training_args = TrainingArguments(
    output_dir="esco_skill_ner_multi_model",               # Output directory for model checkpoints
    learning_rate=2e-4,                              # Standard small learning rate for fine-tuning
    num_train_epochs=10,                              # Number of training epochs 
    per_device_train_batch_size=16,                  # Batch size per GPU/CPU
    per_device_eval_batch_size=16,                   # Evaluation batch size
    weight_decay=0.01,                               # Simple regularization
    eval_strategy="epoch",                     # Evaluate at the end of each epoch
    save_strategy="epoch",                           # Save checkpoint at the end of each epoch
    logging_strategy="epoch",
    load_best_model_at_end=True,                     # Load the model with the best validation performance
    metric_for_best_model="f1",
    push_to_hub=False,                               # Set to True if you want to upload the model to Hugging Face Hub
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 2. Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Ferma se l'F1 non migliora per 3 epoche
)

# 3. Start Training!
print("Starting fine-tuning...")
trainer.train()

print("Fine-tuning complete. Best model saved.")

C:\Users\ACER-PC\AppData\Local\Temp\ipykernel_11008\4168311197.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Starting fine-tuning...


C:\Users\ACER-PC\Desktop\WUIR-CLASS-recSys\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.036400,0.767519,0.238095,0.316456,0.271739,0.771252


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.036400,0.767519,0.238095,0.316456,0.271739,0.771252
2,0.671300,0.618561,0.333333,0.417722,0.370787,0.797527
3,0.450100,0.487662,0.527473,0.607595,0.564706,0.865533
4,0.270000,0.453749,0.547368,0.658228,0.597701,0.873261
5,0.161300,0.403918,0.572917,0.696203,0.628571,0.888717
6,0.103800,0.423290,0.602151,0.708861,0.651163,0.897991
7,0.071000,0.440431,0.584270,0.658228,0.619048,0.891808


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.036400,0.767519,0.238095,0.316456,0.271739,0.771252
2,0.671300,0.618561,0.333333,0.417722,0.370787,0.797527
3,0.450100,0.487662,0.527473,0.607595,0.564706,0.865533
4,0.270000,0.453749,0.547368,0.658228,0.597701,0.873261
5,0.161300,0.403918,0.572917,0.696203,0.628571,0.888717
6,0.103800,0.423290,0.602151,0.708861,0.651163,0.897991
7,0.071000,0.440431,0.584270,0.658228,0.619048,0.891808
8,0.054500,0.437033,0.551020,0.683544,0.610169,0.890263


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.036400,0.767519,0.238095,0.316456,0.271739,0.771252
2,0.671300,0.618561,0.333333,0.417722,0.370787,0.797527
3,0.450100,0.487662,0.527473,0.607595,0.564706,0.865533
4,0.270000,0.453749,0.547368,0.658228,0.597701,0.873261
5,0.161300,0.403918,0.572917,0.696203,0.628571,0.888717
6,0.103800,0.423290,0.602151,0.708861,0.651163,0.897991
7,0.071000,0.440431,0.584270,0.658228,0.619048,0.891808
8,0.054500,0.437033,0.551020,0.683544,0.610169,0.890263
9,0.040800,0.458065,0.602041,0.746835,0.666667,0.894900


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.036400,0.767519,0.238095,0.316456,0.271739,0.771252
2,0.671300,0.618561,0.333333,0.417722,0.370787,0.797527
3,0.450100,0.487662,0.527473,0.607595,0.564706,0.865533
4,0.270000,0.453749,0.547368,0.658228,0.597701,0.873261
5,0.161300,0.403918,0.572917,0.696203,0.628571,0.888717
6,0.103800,0.423290,0.602151,0.708861,0.651163,0.897991
7,0.071000,0.440431,0.584270,0.658228,0.619048,0.891808
8,0.054500,0.437033,0.551020,0.683544,0.610169,0.890263
9,0.040800,0.458065,0.602041,0.746835,0.666667,0.894900
10,0.031800,0.459999,0.585859,0.734177,0.651685,0.894900


Fine-tuning complete. Best model saved.


In [36]:
history = trainer.state.log_history

epochs = sorted({h["epoch"] for h in history if "epoch" in h})

rows = []
for e in epochs:
    # tutte le righe di training di quell’epoch
    train_losses = [h["loss"] for h in history 
                    if h.get("epoch") == e and "loss" in h]
    train_loss = float(np.mean(train_losses)) if train_losses else None

    # una riga di eval per epoch (ce n’è una per "eval_loss")
    eval_entry = next(
        (h for h in history 
         if h.get("epoch") == e and "eval_loss" in h),
        {}
    )

    row = {
        "Epoch": int(e),
        "Training Loss": train_loss,
        "Validation Loss": eval_entry.get("eval_loss"),
        "Precision": eval_entry.get("eval_precision"),
        "Recall": eval_entry.get("eval_recall"),
        "F1": eval_entry.get("eval_f1"),
        "Accuracy": eval_entry.get("eval_accuracy"),
    }
    rows.append(row)

df_metrics = pd.DataFrame(rows).sort_values("Epoch")

df_metrics.head()

,Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
0,1,1.0364,0.767519,0.238095,0.316456,0.271739,0.771252
1,2,0.6713,0.618561,0.333333,0.417722,0.370787,0.797527
2,3,0.4501,0.487662,0.527473,0.607595,0.564706,0.865533
3,4,0.2700,0.453749,0.547368,0.658228,0.597701,0.873261
4,5,0.1613,0.403918,0.572917,0.696203,0.628571,0.888717


In [37]:
predictions, labels, _ = trainer.predict(tokenized_test_dataset)
predictions = np.argmax(predictions, axis=2)

y_true = [
    [id2label[l] for l in label if l != -100]
    for label in labels
]
y_pred = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]


C:\Users\ACER-PC\Desktop\WUIR-CLASS-recSys\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [27]:
def ids_to_tag_seqs(
    y_true: np.ndarray, 
    y_pred: np.ndarray, 
    id2label: dict
) -> Tuple[List[List[str]], List[List[str]]]:
    """
    Converte matrici [num_sentences, max_len] di ID in liste di liste di tag stringa,
    rimuovendo i token con label -100 (padding, subword, ecc.).
    """
    true_seqs = []
    pred_seqs = []

    for true_row, pred_row in zip(y_true, y_pred):
        seq_true = []
        seq_pred = []
        for t_id, p_id in zip(true_row, pred_row):
            if t_id == -100:   # ignora token
                continue
            seq_true.append(id2label[int(t_id)])
            seq_pred.append(id2label[int(p_id)])
        true_seqs.append(seq_true)
        pred_seqs.append(seq_pred)

    return true_seqs, pred_seqs


In [28]:
def plot_classification_report(
    y_pred: np.ndarray,
    y_true: np.ndarray,
    model_name: str,
    ax,
    id2label: dict
):
    """
    Classification report entity-level con seqeval, visualizzato come heatmap.
    """
    # 1) ID → BIO tag sequences
    true_seqs, pred_seqs = ids_to_tag_seqs(y_true, y_pred, id2label)

    # 2) classification_report di seqeval in forma dict
    #    output_dict=True → struttura simile a sklearn (per-classe + avg)
    cr_dict = classification_report(
        true_seqs,
        pred_seqs,
        output_dict=True,
        zero_division=0
    )

    # 3) DataFrame
    cr = pd.DataFrame(cr_dict).T

    # opzionale: tieni solo colonne numeriche principali
    keep_cols = [c for c in ["precision", "recall", "f1-score", "support"] if c in cr.columns]
    cr = cr[keep_cols].astype(float)

    # 4) Heatmap
    sns.heatmap(
        cr,
        annot=True,
        fmt=".2f",
        cmap="Greens",
        cbar=False,
        xticklabels=cr.columns,
        yticklabels=cr.index,
        linewidths=0.6,
        linecolor="black",
        ax=ax,
    )

    ax.set_facecolor("white")
    ax.set_title(f"CR (seqeval) {model_name}", fontsize=14, weight="bold")
    ax.tick_params(axis="x", labelsize=10, rotation=45)
    ax.tick_params(axis="y", labelsize=10)

In [11]:
plot_classification_report(y_pred, y_true, "NER_model", 0, )

NameError: name 'y_pred' is not defined

In [38]:
print(classification_report(y_true=y_true, y_pred=y_pred))

                precision    recall  f1-score   support

ACQUIRED_SKILL       0.53      0.67      0.59        15
   AVOID_SKILL       0.88      1.00      0.93         7
 INCLUDE_SKILL       0.48      0.54      0.51        57
 NEUTRAL_SKILL       0.42      0.57      0.48        14

     micro avg       0.51      0.60      0.55        93
     macro avg       0.58      0.70      0.63        93
  weighted avg       0.51      0.60      0.55        93



In [39]:
import itertools
from sklearn.metrics import confusion_matrix
import pandas as pd

# Flatten
y_true_flat = list(itertools.chain.from_iterable(y_true))
y_pred_flat = list(itertools.chain.from_iterable(y_pred))

def to_binary(label):
    # adattalo ai tuoi label (es. se hai "B-SKILL", "I-SKILL", ecc.)
    return "SKILL" if "SKILL" in label else "OTHER"

y_true_bin = [l for l in y_true_flat]
y_pred_bin = [l for l in y_pred_flat]

'''cm = confusion_matrix(y_true_bin, y_pred_bin, labels=["B-SKILL","I-SKILL", "O"])
df_cm = pd.DataFrame(cm, index=["B-SKILL_true", "I-SKILL_true", "O_true"],
                        columns=["B-SKILL_pred", "I-SKILL_pred", "O_pred"])
print(df_cm)'''

labels_cm = sorted(set(y_true_flat + y_pred_flat))
cm = confusion_matrix(y_true_flat, y_pred_flat, labels=labels_cm)
df_cm = pd.DataFrame(cm, index=[f"{l}_true" for l in labels_cm],
                        columns=[f"{l}_pred" for l in labels_cm])

print(df_cm)


                       B-ACQUIRED_SKILL_pred  B-AVOID_SKILL_pred  \
B-ACQUIRED_SKILL_true                     15                   0   
B-AVOID_SKILL_true                         0                   5   
B-INCLUDE_SKILL_true                       0                   0   
B-NEUTRAL_SKILL_true                       1                   0   
I-ACQUIRED_SKILL_true                      1                   0   
I-AVOID_SKILL_true                         0                   0   
I-INCLUDE_SKILL_true                       0                   0   
I-NEUTRAL_SKILL_true                       0                   0   
O_true                                     1                   1   

                       B-INCLUDE_SKILL_pred  B-NEUTRAL_SKILL_pred  \
B-ACQUIRED_SKILL_true                     0                     0   
B-AVOID_SKILL_true                        0                     0   
B-INCLUDE_SKILL_true                     43                     3   
B-NEUTRAL_SKILL_true                      3

In [40]:

error_examples = []
for i, (tl, tp) in enumerate(zip(y_true, y_pred)):
    if tl != tp:  
        error_examples.append((i, tl, tp))
        
for idx, tl, tp in error_examples[:5]:
    print(f"\nExample {idx}")
    print("TRUE: ", tl)
    print("PRED: ", tp)
    print(test_dataset[idx]["tokens"]) 



Example 0
TRUE:  ['O', 'O', 'O', 'B-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'O', 'O', 'O', 'O', 'O', 'B-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'I-INCLUDE_SKILL']
PRED:  ['O', 'O', 'O', 'B-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'O', 'O', 'O', 'O', 'O', 'I-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'O']
['I', 'already', 'understand', 'marketing', 'funnels', 'but', 'I', 'need', 'to', 'master', 'A/B', 'experimentation', 'frameworks.']

Example 2
TRUE:  ['O', 'O', 'O', 'O', 'O', 'O', 'B-AVOID_SKILL', 'I-AVOID_SKILL', 'I-AVOID_SKILL', 'O', 'O', 'O', 'O', 'O', 'O', 'B-INCLUDE_SKILL', 'I-INCLUDE_SKILL']
PRED:  ['O', 'O', 'O', 'O', 'O', 'O', 'I-AVOID_SKILL', 'I-AVOID_SKILL', 'I-AVOID_SKILL', 'O', 'O', 'O', 'O', 'O', 'O', 'B-INCLUDE_SKILL', 'I-INCLUDE_SKILL']
['I', 'want', 'to', 'avoid', 'using', 'outdated', 'machine', 'control', 'systems,', 'but', 'I', 'need', 'to', 'learn', 'modern', 'PLC', 'programming.']

Example 5
TRUE:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'I-INCLUDE_S